<!-- project-navigation -->
# 04_data_understanding_installments_payments
Role: Exploration and data preparation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [1]:
import pandas as pd


# Chargement
installments = pd.read_csv(
    "data/raw/installments_payments.csv"
)


print("========== DIMENSION ==========")
print(installments.shape)


print("\n========== INFORMATIONS GENERALES ==========")
print(installments.info())


print("\n========== TYPES DES VARIABLES ==========")
print(installments.dtypes.value_counts())


print("\n========== VARIABLES ==========")
print(installments.columns.tolist())


print("\n========== VALEURS MANQUANTES ==========")

missing = installments.isna().sum()

missing_percent = (missing / len(installments)) * 100

missing_table = pd.DataFrame({
    "Nombre_manquant": missing,
    "Pourcentage": missing_percent
})


display(
    missing_table[missing_table["Nombre_manquant"] > 0]
    .sort_values("Pourcentage", ascending=False)
)


print("\n========== APERCU ==========")

display(installments.head())


print("\n========== STATISTIQUES NUMERIQUES ==========")

display(
    installments.describe().T
)

========== DIMENSION ==========
(13605401, 8)

========== INFORMATIONS GENERALES ==========
<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB
None

========== TYPES DES VARIABLES ==========
float64    5
int64      3
Name: count, dtype: int64

========== VARIABLES ==========
['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']

========== VALEURS MANQUANTES ==========


,Nombre_manquant,Pourcentage
DAYS_ENTRY_PAYMENT,2905,0.021352
AMT_PAYMENT,2905,0.021352



========== APERCU ==========


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585



========== STATISTIQUES NUMERIQUES ==========


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,13605401.0,1.903365e+06,536202.905546,1000001.0,1434191.000,1896520.000,2369094.000,2843499.000
SK_ID_CURR,13605401.0,2.784449e+05,102718.310411,100001.0,189639.000,278685.000,367530.000,456255.000
NUM_INSTALMENT_VERSION,13605401.0,8.566373e-01,1.035216,0.0,0.000,1.000,1.000,178.000
NUM_INSTALMENT_NUMBER,13605401.0,1.887090e+01,26.664067,1.0,4.000,8.000,19.000,277.000
DAYS_INSTALMENT,13605401.0,-1.042270e+03,800.946284,-2922.0,-1654.000,-818.000,-361.000,-1.000
DAYS_ENTRY_PAYMENT,13602496.0,-1.051114e+03,800.585883,-4921.0,-1662.000,-827.000,-370.000,-1.000
AMT_INSTALMENT,13605401.0,1.705091e+04,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845
AMT_PAYMENT,13602496.0,1.723822e+04,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845


# Analyse et transformation de la table `installments_payments`

## 1. Objectif de la table

La table `installments_payments` contient l'historique des remboursements effectués par les clients.

Contrairement aux tables :

- `application_train` : 1 ligne = 1 client ;
- `bureau` : 1 ligne = 1 crédit historique ;

la table `installments_payments` est au niveau paiement :


1 ligne = 1 échéance de remboursement


Un même client peut donc apparaître plusieurs fois car il peut avoir plusieurs crédits et plusieurs échéances.

---

# 2. Structure initiale

La table contient :


13 605 401 lignes
8 variables


Chaque observation représente un remboursement associé à un crédit.

Les principales variables sont :

| Variable | Description |
|-|-|
| SK_ID_PREV | Identifiant du crédit précédent |
| SK_ID_CURR | Identifiant du client |
| NUM_INSTALMENT_NUMBER | Numéro de l'échéance |
| DAYS_INSTALMENT | Date prévue du paiement |
| DAYS_ENTRY_PAYMENT | Date réelle du paiement |
| AMT_INSTALMENT | Montant prévu |
| AMT_PAYMENT | Montant réellement payé |

---

# 3. Analyse des valeurs manquantes

L'analyse montre des valeurs manquantes uniquement dans :

- `DAYS_ENTRY_PAYMENT`
- `AMT_PAYMENT`

Ces deux variables représentent un paiement réellement effectué.

Une valeur manquante signifie généralement qu'une information de paiement n'a pas été enregistrée.

La stratégie choisie est :

- conserver les lignes concernées ;
- créer des indicateurs permettant de détecter les paiements incomplets ;
- remplacer les valeurs manquantes par 0 uniquement pour les calculs nécessaires.

Justification métier :

Une absence de paiement enregistré peut représenter un comportement de remboursement différent d'un paiement normal.

---

# 4. Analyse du comportement de remboursement

Cette table permet de créer des indicateurs liés au risque :

- respect des échéances ;
- retard de paiement ;
- montant réellement remboursé ;
- différence entre montant prévu et montant payé.

---

# 5. Création des variables comportementales

La table originale contient des informations détaillées par échéance.

Elle sera transformée en variables synthétiques au niveau client.

Transformation :


installments_payments

13 605 401 lignes
1 ligne = 1 paiement

    ↓

Agrégation par SK_ID_CURR

    ↓

1 ligne = 1 client


---

# 6. Variables créées

## Nombre total d'échéances

Variable :


NB_INSTALLMENTS


Elle représente le nombre total de paiements historiques du client.

Elle permet d'évaluer l'expérience de remboursement.

---

## Montant total prévu

Variable :


TOTAL_INSTALLMENT_AMOUNT


Calcul :

Somme des montants prévus :

AMT_INSTALMENT

Elle représente l'engagement financier total du client.

---

## Montant total payé

Variable :


TOTAL_PAYMENT_AMOUNT


Calcul :

Somme des paiements réellement effectués :

AMT_PAYMENT

---

## Différence entre montant prévu et montant payé

Variable :


TOTAL_PAYMENT_DIFF


Calcul :


AMT_INSTALMENT - AMT_PAYMENT


Une différence élevée peut indiquer des difficultés de remboursement.

---

## Nombre de paiements en retard

Pour chaque paiement :


RETARD =
DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT


Si :


RETARD > 0


le client a payé après la date prévue.

Variable créée :


NB_LATE_PAYMENTS


---

## Retard moyen

Variable :


AVG_PAYMENT_DELAY


Elle mesure le retard moyen observé chez le client.

---

## Ratio de paiement

Variable :


PAYMENT_RATIO


Calcul :


TOTAL_PAYMENT_AMOUNT /
TOTAL_INSTALLMENT_AMOUNT


Interprétation :

- proche de 1 : paiement complet ;
- inférieur à 1 : paiements incomplets.

---

# 7. Niveau final attendu

Après transformation :

Avant :


13 605 401 lignes

1 ligne = 1 remboursement


Après :


Nombre de clients

1 ligne = 1 client


Les nouvelles variables représentent le comportement réel de remboursement.

---

# 8. Position dans le pipeline CRISP-DM

Cette étape appartient à :

## Data Understanding

- comprendre les variables ;
- identifier leur signification métier ;
- analyser les valeurs manquantes.

## Data Preparation

- créer les indicateurs comportementaux ;
- agréger au niveau client ;
- préparer les données pour le Machine Learning.

---

# 9. Prochaine étape

Après cette transformation, les trois sources historiques seront prêtes :

1. `bureau_features`
2. `bureau_balance_features`
3. `installments_features`

Elles seront ensuite fusionnées avec :


application_train


pour construire le dataset final de modélisation du risque de crédit.

In [2]:
# ============================================================
# Transformation installments_payments au niveau client
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1) Chargement
# ------------------------------------------------------------

installments = pd.read_csv(
    "data/raw/installments_payments.csv"
)

print("Dimension initiale :", installments.shape)


# ------------------------------------------------------------
# 2) Création des variables métier
# ------------------------------------------------------------

print("\nCréation des variables comportementales...")


# Remplacement temporaire pour calculs
installments["DAYS_ENTRY_PAYMENT"] = installments["DAYS_ENTRY_PAYMENT"].fillna(
    installments["DAYS_INSTALMENT"]
)

installments["AMT_PAYMENT"] = installments["AMT_PAYMENT"].fillna(0)


# Retard de paiement en jours

installments["PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    -
    installments["DAYS_INSTALMENT"]
)


# Paiement en retard

installments["IS_LATE"] = (
    installments["PAYMENT_DELAY"] > 0
).astype(int)


# Différence entre montant prévu et payé

installments["PAYMENT_DIFF"] = (
    installments["AMT_INSTALMENT"]
    -
    installments["AMT_PAYMENT"]
)


# Paiement complet ou incomplet

installments["IS_UNDERPAID"] = (
    installments["PAYMENT_DIFF"] > 0
).astype(int)



# ------------------------------------------------------------
# 3) Agrégation par client
# ------------------------------------------------------------

installments_features = installments.groupby("SK_ID_CURR").agg(

    # Nombre total d'échéances
    NB_INSTALLMENTS = (
        "SK_ID_PREV",
        "count"
    ),

    # Nombre de crédits concernés
    NB_PREVIOUS_CREDITS = (
        "SK_ID_PREV",
        "nunique"
    ),

    # Montants
    TOTAL_INSTALLMENT_AMOUNT = (
        "AMT_INSTALMENT",
        "sum"
    ),

    TOTAL_PAYMENT_AMOUNT = (
        "AMT_PAYMENT",
        "sum"
    ),

    # Retards
    NB_LATE_PAYMENTS = (
        "IS_LATE",
        "sum"
    ),

    AVG_PAYMENT_DELAY = (
        "PAYMENT_DELAY",
        "mean"
    ),

    MAX_PAYMENT_DELAY = (
        "PAYMENT_DELAY",
        "max"
    ),

    # Sous-paiements
    NB_UNDERPAID = (
        "IS_UNDERPAID",
        "sum"
    ),

    AVG_PAYMENT_DIFF = (
        "PAYMENT_DIFF",
        "mean"
    )

).reset_index()



# ------------------------------------------------------------
# 4) Création des ratios
# ------------------------------------------------------------


installments_features["PAYMENT_RATIO"] = (

    installments_features["TOTAL_PAYMENT_AMOUNT"]
    /
    installments_features["TOTAL_INSTALLMENT_AMOUNT"]

)


installments_features["LATE_PAYMENT_RATIO"] = (

    installments_features["NB_LATE_PAYMENTS"]
    /
    installments_features["NB_INSTALLMENTS"]

)



# ------------------------------------------------------------
# 5) Traitement des valeurs manquantes
# ------------------------------------------------------------

installments_features = installments_features.replace(
    [np.inf, -np.inf],
    np.nan
)


installments_features = installments_features.fillna(0)



# ------------------------------------------------------------
# 6) Vérification finale
# ------------------------------------------------------------

print("\n========== TABLE FINALE ==========")

print(installments_features.shape)


print("\nValeurs manquantes :")

print(
    installments_features.isna().sum().sum()
)


print("\nAperçu :")

display(
    installments_features.head()
)



# ------------------------------------------------------------
# 7) Sauvegarde
# ------------------------------------------------------------

installments_features.to_csv(
    "data/processed/installments_features.csv",
    index=False
)


print(
    "\nFichier sauvegardé : installments_features.csv"
)

Dimension initiale : (13605401, 8)

Création des variables comportementales...

========== TABLE FINALE ==========
(339587, 12)

Valeurs manquantes :
0

Aperçu :


,SK_ID_CURR,NB_INSTALLMENTS,NB_PREVIOUS_CREDITS,TOTAL_INSTALLMENT_AMOUNT,TOTAL_PAYMENT_AMOUNT,NB_LATE_PAYMENTS,AVG_PAYMENT_DELAY,MAX_PAYMENT_DELAY,NB_UNDERPAID,AVG_PAYMENT_DIFF,PAYMENT_RATIO,LATE_PAYMENT_RATIO
0,100001,7,2,41195.925,41195.925,1,-7.285714,11.0,0,0.0,1.0,0.142857
1,100002,19,1,219625.695,219625.695,0,-20.421053,-12.0,0,0.0,1.0,0.000000
2,100003,25,3,1618864.650,1618864.650,0,-7.160000,-1.0,0,0.0,1.0,0.000000
3,100004,3,1,21288.465,21288.465,0,-7.666667,-3.0,0,0.0,1.0,0.000000
4,100005,9,1,56161.845,56161.845,1,-23.555556,1.0,0,0.0,1.0,0.111111



Fichier sauvegardé : installments_features.csv


# Transformation de la table `installments_payments` au niveau client

## 1. Objectif

La table `installments_payments` contient l'historique détaillé des remboursements effectués par les clients.

Avant transformation :


13 605 401 lignes
1 ligne = 1 échéance de remboursement


Cette structure n'est pas directement adaptée au Machine Learning car la variable cible `TARGET` est définie au niveau client.

L'objectif est donc de transformer les paiements individuels en indicateurs synthétiques représentant le comportement de remboursement de chaque client.

---

# 2. Création des variables comportementales

## Calcul du retard de paiement

Une nouvelle variable a été créée :


PAYMENT_DELAY


Elle correspond à :


Date réelle du paiement - Date prévue du paiement


Interprétation :

- valeur positive → paiement effectué en retard ;
- valeur négative → paiement effectué avant l'échéance.

Exemple :

| DAYS_INSTALMENT | DAYS_ENTRY_PAYMENT | PAYMENT_DELAY |
|-|-|-|
| -100 | -95 | 5 |

Le client a payé avec 5 jours de retard.

---

# 3. Identification des paiements en retard

Variable créée :


IS_LATE


Règle :


IS_LATE = 1 si PAYMENT_DELAY > 0
IS_LATE = 0 sinon


Cette variable permet de compter le nombre d'incidents de paiement.

---

# 4. Analyse des différences de paiement

Une variable supplémentaire a été créée :


PAYMENT_DIFF


Calcul :


AMT_INSTALMENT - AMT_PAYMENT


Elle mesure l'écart entre :

- le montant prévu ;
- le montant réellement payé.

Un écart élevé peut indiquer un remboursement incomplet.

---

# 5. Agrégation au niveau client

La transformation a été réalisée avec :


GROUP BY SK_ID_CURR


afin d'obtenir :


1 ligne = 1 client


Les variables créées sont :

| Variable | Signification |
|-|-|
| NB_INSTALLMENTS | Nombre total d'échéances historiques |
| NB_PREVIOUS_CREDITS | Nombre de crédits concernés |
| TOTAL_INSTALLMENT_AMOUNT | Montant total prévu |
| TOTAL_PAYMENT_AMOUNT | Montant total payé |
| NB_LATE_PAYMENTS | Nombre de paiements en retard |
| AVG_PAYMENT_DELAY | Retard moyen |
| MAX_PAYMENT_DELAY | Retard maximal observé |
| NB_UNDERPAID | Nombre de paiements inférieurs au montant prévu |
| AVG_PAYMENT_DIFF | Différence moyenne entre prévu et payé |

---

# 6. Création des ratios

## Ratio global de paiement

Variable :


PAYMENT_RATIO


Calcul :


TOTAL_PAYMENT_AMOUNT /
TOTAL_INSTALLMENT_AMOUNT


Interprétation :

- proche de 1 : remboursement complet ;
- inférieur à 1 : paiements incomplets.

---

## Ratio de retard

Variable :


LATE_PAYMENT_RATIO


Calcul :


Nombre de paiements en retard /
Nombre total de paiements


Cette variable permet de comparer les clients ayant des historiques différents.

---

# 7. Traitement des valeurs manquantes

Les valeurs manquantes concernaient :

- DAYS_ENTRY_PAYMENT ;
- AMT_PAYMENT.

Traitement appliqué :

## DAYS_ENTRY_PAYMENT

Remplacement par :


DAYS_INSTALMENT


Justification :

Une absence de date de paiement indique qu'aucun écart de date ne peut être calculé.

Cela évite de créer artificiellement un retard.

---

## AMT_PAYMENT

Remplacement par :


0


Justification :

Une absence de montant payé est considérée comme une absence de paiement enregistré.

---

Après traitement :


Valeurs manquantes restantes = 0


---

# 8. Résultat final

Avant transformation :


13 605 401 lignes
1 ligne = 1 remboursement


Après transformation :


339 587 lignes
1 ligne = 1 client


La table finale contient :


12 variables comportementales


représentant l'historique de remboursement des clients.

---

# 9. Interprétation de quelques résultats

Exemple client :


SK_ID_CURR = 100001

NB_INSTALLMENTS = 7
PAYMENT_RATIO = 1
LATE_PAYMENT_RATIO = 0.14


Interprétation :

- le client possède 7 remboursements historiques ;
- la totalité des montants prévus a été payée ;
- environ 14% des paiements ont présenté un retard.

---

# 10. Position dans le pipeline CRISP-DM

Cette étape appartient à :

## Data Understanding

- analyse de la structure ;
- compréhension métier des variables.

## Data Preparation

- création des variables comportementales ;
- agrégation client ;
- préparation pour la modélisation.

---

# 11. Prochaine étape

Les tables historiques sont maintenant préparées :

✓ application_train  
✓ bureau_features  
✓ bureau_balance_features  
✓ installments_features  

La prochaine étape est la fusion finale :


application_train

    +

bureau_features

    +

bureau_balance_features

    +

installments_features

    ↓

Dataset final pour Machine Learning


Ce dataset sera utilisé ensuite pour :

- encodage final ;
- sélection des variables ;
- séparation train/test ;
- entraînement des modèles de classification :
  - régression logistique ;
  - Random Forest ;
  - modèles Boosting.
Remarque importante sur ton résultat :

Tu obtiens :

installments_features : (339 587, 12)

alors que application_train contient :

307 511 clients

C'est normal.

Pourquoi ?

Parce que installments_payments contient aussi des clients présents dans application_test ou des clients qui n'ont pas de ligne dans application_train.